# A4 — Ultralytics YOLO Experiment Notebook

This notebook is the **primary interface** of an A4 project (see the repository
`README.md`). Every project under `projects/` is an edited instance of this notebook

The notebook is a **guided, executable experiment**, not a bag of independent code
snippets. Markdown cells explain *why* a step exists and *what breaks if you skip it*;
code cells do the work by delegating almost everything to **Ultralytics** (the package
and, optionally, the Platform). In practice, most failed YOLO projects fail for one of
two boring reasons: a dataset problem that was never caught before training started, or
an experiment that can't be reproduced later because nobody recorded what was actually
run. This notebook is structured specifically to make both mistakes hard to make.

## The seven stages

| Stage | Name | Purpose |
|---|---|---|
| 0 | Environment | Install / verify Ultralytics, GPU, paths |
| 1 | Experiment Setup | Declare `TASK`, `MODEL`, `DATA`, experiment name, overrides |
| 2 | Dataset Preparation & Validation | Structural check → visual spot-check → smoke test |
| 3 | Model & Training | Load pretrained weights, optional class-alias transfer, train, optional tuning |
| 4 | Evaluation & Inspection | Evaluate `best.pt`, read the plots, look at predictions |
| 5 | Experiment Summary | Assemble config + metrics into one readable record |
| 6 | Export | Convert to a deployment format and re-validate |
| 7 | Report | Write a durable markdown report under `docs/` |

Most of the time you only touch a handful of variables in Section 1 and then run the
notebook top to bottom. Custom code should stay rare — if Ultralytics (or Ultralytics
Platform) already does it, use that instead of writing a second, possibly buggy, version
of the same logic.

## Turning this template into a real project

You never start a project from an empty folder — you copy this template and edit only
the parts that must differ for your task and dataset:

1. **Choose a task and a dataset name.** Task must be one of `detect`, `segment`,
   `semantic`, `depth`, `classify`, `pose`, `obb`. Dataset is a short name such as `VOC`,
   `MyCars`, `WarehouseBoxes`. The target path is `projects/<Task>/<Dataset>/`
   (e.g. `projects/Detect/VOC/`).
2. Copy the entire template tree, not just this notebook, by running the following command in your terminal. Replace "Detect" with your desired task and "MyDataset_Project" with your project name, which should include the dataset name.

```bash
   git clone https://github.com/AmirMahdiRezaeiEECS/ALL-IN-ONE-VISION-4
   cd ALL-IN-ONE-VISION-4
   cp -r template projects/Detect/MyDataset_Project
```
3. **Add a dataset YAML** under `configs/datasets/<Dataset>.yaml` (or a class-per-folder
   layout for classification — see the `yolo-datasets` skill). If `path:` inside it is
   relative, remember it resolves against `datasets_dir` (Section 0 below).
4. **Edit the five identity variables** in Section 1 — `TASK`, `MODEL`, `DATA`,
   `EXPERIMENT_NAME`, `PROJECT` — and, only if you have a specific reason to deviate from
   Ultralytics' defaults, add a matching `configs/experiments/<EXPERIMENT_NAME>.yaml`.
5. **Run top to bottom**, in order, the first time: Environment → Setup (watch the
   suffix-mismatch warning) → Dataset checks (set `RUN_SMOKE_TEST = True` for a new or
   changed dataset) → Train (optionally set `class_aliases` first if the pretrained
   checkpoint and your dataset name shared classes differently) → Evaluate `best.pt` →
   Summary → Export (if needed) → Report.
6. **Add later experiments to the same project** — do not create a new project folder
   per model size or hyperparameter change. Change `MODEL` and/or the overrides file,
   pick a new self-describing `EXPERIMENT_NAME`, and re-run. Every run gets its own
   directory under `runs/`, so nothing is overwritten and everything stays comparable.

A fuller checklist and the full recipe live in `docs/02_how_to_edit_the_template.md`; a
section-by-section explanation of every cell below lives in
`docs/03_notebook_walkthrough.md`. This notebook's markdown cells are the condensed,
in-context version of both.

## Optional — Transfer Classes with Name Aliases

Ultralytics transfers pretrained classification-head rows to classes with a
matching name (case/whitespace-insensitive) — this is `cls_remap`, on by default.
If your dataset names a shared concept differently than the pretrained checkpoint
(COCO `airplane` vs. VOC `aeroplane`), that row is treated as unmatched and
reinitialized at random, silently discarding pretrained signal.

`CLASS_ALIASES` in Section 3 renames the checkpoint's class names in memory,
right before training, so the transfer succeeds for those classes too. Like
tuning, its value is **entirely experiment-dependent** — it helps only for the
specific checkpoint + dataset pair at hand, so it gets the same baseline-first
discipline instead of being turned on by default:

1. Train a `baseline` experiment with no `class_aliases` at all.
2. Compare the pretrained checkpoint's class names against your dataset's class
   names (Section 2's optional discovery cell, and the class list already
   printed in Section 2) to find candidates.
3. Record a `class_aliases` mapping in a **new** `configs/experiments/<name>.yaml`
   and train a second experiment (e.g. `baseline_aliased`) with that
   `EXPERIMENT_NAME`.
4. Compare **per-class** metrics in both runs' confusion matrices (Section 4) —
   only the aliased classes should move. Keep the aliased experiment only if it
   demonstrably helps; otherwise the baseline stands.

See Section 3 for the exact mechanics.

## Optional hyperparameter tuning (baseline-first workflow)

Hyperparameter tuning is **optional and off by default** (`RUN_TUNE = False` in Section 3).
It uses Ultralytics' genetic tuner (`model.tune()`), which runs many short trainings. That
is expensive, so the notebook forces a deliberate order:

1. **Always train a single-shot baseline first** (the normal `model.train(...)` cell).
2. Inspect the baseline in Section 4 (metrics, plots, a few predictions). Fix data,
   labels, `imgsz`, model size, or epoch count before considering tuning.
3. Only then set `RUN_TUNE = True`, choose modest `TUNE_EPOCHS` / `TUNE_ITERATIONS`, and
   run the tuning cell. Results land under `runs/<task>/<name>_tune/best_hyperparameters.yaml`.
4. **Promote** the useful values into a new (or updated) `configs/experiments/<name>.yaml`,
   pick a new self-describing `EXPERIMENT_NAME`, leave `RUN_TUNE = False`, and re-run a
   full-length training. Do not treat the short-search checkpoint as the final model.

Typical realistic gain is 0.5–2 mAP. Exhaust cheaper levers first (data quality → longer
training → larger `imgsz` / model → domain augmentation). See the `yolo-tuning` skill and
[Ultralytics hyperparameter-tuning guide](https://docs.ultralytics.com/guides/hyperparameter-tuning/).

## 0. Environment


In [ ]:
%pip install -q -U ultralytics

Colab/Kaggle often ship an older `ultralytics`. Reinstall with `-U` each session so you pick up upstream fixes.


In [ ]:
import os
from pathlib import Path

import yaml

from ultralytics import YOLO

PROJECT_ROOT = Path.cwd()
print(f"Working directory: {PROJECT_ROOT}")

`yolo checks` reports versions, GPU visibility, and disk space. Read it once per session — silent CPU training is a common time sink.


In [ ]:
!yolo checks

**Optional — Ultralytics Platform.** Set an API key for Platform-hosted data (`ul://...`) and metric streaming (`project=username/slug`). Local `data.yaml` + `runs/` work without it.

`datasets_dir` is important: relative `path:` values in a data YAML resolve against it. A stale setting causes false “Dataset not found” errors. Set it once per environment.


In [ ]:
# os.environ["ULTRALYTICS_API_KEY"] = "..."  # uncomment to enable Platform streaming

# If `path:` in your data.yaml is relative, it resolves against `datasets_dir` below.
# Point it at your actual dataset root once per environment (Colab/Kaggle sessions reset).
# !yolo settings datasets_dir=/content/datasets

## 1. Experiment Setup

**Purpose.** Declare task, model, dataset, experiment name, and overrides in one place so the run is reproducible later.

- **TASK** selects the model head, loss, and label format. A mismatch with the checkpoint suffix trains “successfully” but learns nothing useful — that is why the sanity-check cell exists.
- **Overrides** should be deliberate and recorded (e.g. “raised `imgsz` for small objects”), not leftover defaults.

**Edit:** the five identity variables below, and only if needed a matching `configs/experiments/<EXPERIMENT_NAME>.yaml`.


In [ ]:
# --- Experiment identity -----------------------------------------------------
TASK = "detect"  # detect | segment | semantic | depth | classify | pose | obb
MODEL = "yolo26n.pt"  # ALWAYS a pretrained checkpoint; start with 'n' to validate the
                        # pipeline cheaply, then scale up (see the yolo-models skill)
DATA = "configs/datasets/dataset.yaml"  # local yaml, classify folder, or ul://... URI

EXPERIMENT_NAME = "baseline"  # self-describing, e.g. "0906_yolo26n_voc_e100"
PROJECT = f"runs/{TASK}"       # or "username/project-slug" to stream to Platform

# --- Config-driven overrides ---------------------------------------------------
# Only settings we have a strong reason to change belong here (README: "Default-First
# Configuration"). Everything else is left to Ultralytics.
experiment_config_path = Path(f"configs/experiments/{EXPERIMENT_NAME}.yaml")
overrides = {}
if experiment_config_path.exists():
    with open(experiment_config_path) as f:
        overrides = yaml.safe_load(f) or {}

print(f"Task:      {TASK}")
print(f"Model:     {MODEL}")
print(f"Data:      {DATA}")
print(f"Run:       {PROJECT}/{EXPERIMENT_NAME}")
print(f"Overrides: {overrides}")

**Why this check exists.** Model filenames encode the task (`-seg`, `-pose`, `-obb`, …; no suffix = detect). A mismatch does not hard-error; it often shows up later as near-zero mAP. This cell warns early. It only warns (does not block) because families like YOLO-World, YOLOE, SAM, or RT-DETR do not follow the naming convention.


In [ ]:
_TASK_SUFFIX = {
    "detect": "", "segment": "-seg", "semantic": "-sem", "depth": "-depth",
    "classify": "-cls", "pose": "-pose", "obb": "-obb",
}
_stem = Path(MODEL).stem
_all_suffixes = [s for s in _TASK_SUFFIX.values() if s]
_expected = _TASK_SUFFIX.get(TASK, "")

if _stem.startswith(("yolo",)):  # only applies to plain YOLO family naming
    if _expected and not _stem.endswith(_expected):
        print(f"WARNING: MODEL='{MODEL}' does not end in '{_expected}' for TASK='{TASK}'.")
    elif not _expected and any(_stem.endswith(s) for s in _all_suffixes):
        print(f"WARNING: MODEL='{MODEL}' looks task-suffixed but TASK='{TASK}' (detect) was chosen.")

## 2. Dataset Preparation & Validation

**Purpose.** Catch dataset problems before long GPU runs. Bad YOLO data usually trains without crashing and produces a quietly useless model.

Ordered checks (cheap → conclusive):

1. **Structural** (`check_det_dataset`) — YAML parses, paths exist.
2. **Visual spot-check** (detect) — one label lands on the object.
3. **Smoke test** (`RUN_SMOKE_TEST = True`) — real dataloader, 1 epoch, 10% data.
4. **Distribution plots** — class balance and augmentation coverage (free from the smoke test).

Prefer Ultralytics converters for COCO/DOTA/masks; put one-off scripts under `scripts/data/`.

**Edit:** set `RUN_SMOKE_TEST = True` for a new or changed dataset; leave `False` once the data is trusted.


In [ ]:
from ultralytics.data.utils import check_det_dataset

if TASK != "classify":
    dataset_info = check_det_dataset(DATA)  # validates data.yaml, resolves paths
    names = dataset_info["names"]
    print(f"Classes ({len(names)}): {names}")
else:
    dataset_info = None
    names = None
    print(f"Classification dataset folder: {DATA}")

`check_det_dataset` only validates structure (keys, paths, optional auto-download). It does not open every image or parse every label.

**Visual spot-check** (detect only): labels are `class cx cy w h` with center coordinates. Ultralytics maps image → label by replacing the last `/images/` segment with `/labels/` and the extension with `.txt` (`img2label_paths`). Prefer that over a naive string replace.


In [ ]:
from ultralytics.data.utils import visualize_image_annotations
from ultralytics.data.utils import img2label_paths

if TASK == "detect":
    train_dir = dataset_info["train"]
    train_dir = Path(train_dir[0] if isinstance(train_dir, list) else train_dir)
    sample_image = next(
        (p for ext in ("*.jpg", "*.jpeg", "*.png") for p in train_dir.glob(ext)), None
    )
    sample_label = Path(img2label_paths([str(sample_image)])[0])
    visualize_image_annotations(str(sample_image), str(sample_label), label_map=names)

**Task-loader smoke test.** Builds the real Ultralytics dataset/dataloader on a small slice (`fraction=0.1`, `epochs=1`). Anything that would break real training fails here in under a minute. Set `RUN_SMOKE_TEST = True` once per new dataset or after label changes.


In [ ]:
RUN_SMOKE_TEST = False
SMOKE_DIR = Path(PROJECT) / "smoke_test"

if RUN_SMOKE_TEST:
    YOLO(MODEL).train(
        data=DATA,
        epochs=1,
        fraction=0.1,
        project=PROJECT,
        name="smoke_test",
        exist_ok=True,
    )

Plots produced by the smoke test:

- **`labels.jpg`** — class histogram and box size/position distributions (spot imbalance early).
- **`labels_correlogram.jpg`** — pairwise box-parameter correlations (labeling artifacts).
- **`train_batch0.jpg`** — augmented images with labels drawn; fastest way to catch labeling errors.


In [ ]:
from IPython.display import Image, display

if RUN_SMOKE_TEST:
    # Class balance, aug-space coverage, and label correctness — all from Ultralytics,
    # none of it hand-rolled.
    for plot_name in ("labels.jpg", "labels_correlogram.jpg", "train_batch0.jpg"):
        plot_path = SMOKE_DIR / plot_name
        if plot_path.exists():
            display(Image(filename=str(plot_path)))

**Optional — discover class-name aliasing candidates.** The class list printed
above is this dataset's target names. Run the cell below to also print `MODEL`'s
pretrained class names, then compare the two lists by eye for concepts named
differently (e.g. `airplane` vs. `aeroplane`). This is informational only — it
does not change anything. See Section 3 to act on what you find.


In [ ]:
SHOW_PRETRAINED_CLASSES = False  # set True to list MODEL's pretrained class names

if SHOW_PRETRAINED_CLASSES:
    _pretrained_names = sorted(YOLO(MODEL).model.names.values())
    print(f"Pretrained classes ({len(_pretrained_names)}): {_pretrained_names}")


## 3. Model & Training

**Purpose.** Load a pretrained checkpoint and train.

Start from pretrained weights (e.g. `yolo26n.pt`). They already encode general visual features; fine-tuning adapts them to your classes. Training from random weights needs far more data and is avoided here.

Class-count mismatch is handled automatically: the head is re-initialized; the backbone keeps pretrained weights.

**Edit:** usually only the overrides file and experiment name. If the
pretrained checkpoint and your dataset name a shared class differently, see
the optional aliasing step immediately below before training.


In [ ]:
model = YOLO(MODEL)  # always start from pretrained weights


### Optional — Transfer Classes with Name Aliases

`cls_remap` (Ultralytics-side, on by default) already transfers pretrained
classification-head rows whose names match this dataset's names exactly
(case/whitespace-insensitive). Use `CLASS_ALIASES` only when the checkpoint and
this dataset name the same concept differently — otherwise that row is silently
reinitialized at random instead of reusing pretrained weights.

Define the mapping in `configs/experiments/<EXPERIMENT_NAME>.yaml` under a
`class_aliases` key (source/pretrained name → this dataset's name), e.g.:

```yaml
class_aliases:
  airplane: aeroplane
  motorcycle: motorbike
```

**`MODEL` must stay the untouched, officially pretrained checkpoint for this to do
anything.** Aliasing only has an effect on the *original* pretrained classification
head — its names are the pretrained vocabulary, and its weights still carry
general-purpose features. If `MODEL` instead points at a previously fine-tuned
`best.pt` (e.g. from the `baseline` run), aliasing silently does nothing useful: that
checkpoint's names are already your dataset's names (so no `class_aliases` key
matches), and its weights are already dataset-specific, not the general pretrained
ones this technique is meant to preserve. Keep `MODEL` identical between the
`baseline` and the aliased experiment — only `EXPERIMENT_NAME` and `class_aliases`
should differ.

Leave it empty/absent to skip this step entirely — that is the required first
(`baseline`) run. See the intro section "Optional — Transfer Classes with Name
Aliases" for the full baseline-first workflow.


In [ ]:
# --- Optional: Transfer Classes with Name Aliases -----------------------------
# `class_aliases` is not a model.train() argument, so it is popped out of
# `overrides` here rather than left in (it would otherwise crash model.train()).
CLASS_ALIASES = overrides.pop("class_aliases", {})

if CLASS_ALIASES:
    # Guard: this only works on the ORIGINAL pretrained checkpoint. A MODEL path
    # that lives under a runs/ tree is almost certainly a previously fine-tuned
    # checkpoint (e.g. a prior baseline's best.pt) whose names/weights are already
    # dataset-specific -- aliasing it is a silent no-op at best.
    if "runs" in Path(MODEL).parts:
        print(
            f"WARNING: MODEL='{MODEL}' looks like a fine-tuned checkpoint (path "
            "contains 'runs/'). class_aliases only transfers weights from the "
            "ORIGINAL pretrained checkpoint -- point MODEL at the untouched "
            "official weights (e.g. 'yolo26n.pt'), the same one used for the "
            "unaliased baseline, not a previous run's best.pt."
        )

    target_names = set(names.values()) if names else set()
    unmatched = [v for v in CLASS_ALIASES.values() if target_names and v not in target_names]
    if unmatched:
        print(f"WARNING: alias target(s) not found in dataset classes: {unmatched}")

    model.model.names = {
        i: CLASS_ALIASES.get(n.lower(), n) for i, n in model.model.names.items()
    }
    print(f"Applied {len(CLASS_ALIASES)} class-name alias(es) before training.")


In [ ]:
model.train(
    data=DATA,
    project=PROJECT,
    name=EXPERIMENT_NAME,
    **overrides,
)

RUN_DIR = Path(model.trainer.save_dir)
print(f"Run saved to: {RUN_DIR}")


Each run gets its own directory under `runs/` (name may be auto-suffixed if taken). `RUN_DIR` records the actual path.

**Optional — hyperparameter tuning** (off by default).

Ultralytics uses a genetic algorithm that mutates learning-rate, loss gains and augmentation hyperparameters across many *short* training runs. Realistic gain is usually only 0.5–2 mAP. Follow the improvement order in the `yolo-tuning` skill first (data → longer training → larger `imgsz` / model → domain aug → *then* tune).

Leave `RUN_TUNE = False` until the baseline in Section 4 looks healthy. When you enable it, the search writes `best_hyperparameters.yaml`; you then promote the useful values into a new experiment config and retrain fully.(see the intro section
"Optional hyperparameter tuning (baseline-first workflow)").

In [ ]:
# --- Optional hyperparameter tuning (disabled by default) --------------------
# Genetic search. Each iteration = one short training. Expensive.
# Enable only after the baseline above has been inspected in Section 4.
RUN_TUNE = False

TUNE_EPOCHS     = 30          # short runs for the search
TUNE_ITERATIONS = 50          # start modest (100–300 is typical)
TUNE_NAME       = f"{EXPERIMENT_NAME}_tune"

if RUN_TUNE:
    tune_model = YOLO(MODEL)
    tune_model.tune(
        data=DATA,
        epochs=TUNE_EPOCHS,
        iterations=TUNE_ITERATIONS,
        project=PROJECT,
        name=TUNE_NAME,
        plots=False,
        save=False,
        val=False,
        # space={...}          # optional custom search space
        # use_ray=True,        # only if you installed ray[tune] and want parallel trials
        # resume=True,         # if a previous tune was interrupted
    )
    print("Tuning finished.")
    print(f"Best hyperparameters → runs/{TASK}/{TUNE_NAME}/best_hyperparameters.yaml")
    print("Copy the values you want into configs/experiments/<new-name>.yaml")
    print("then set a new EXPERIMENT_NAME and re-run the normal training cell.")

## 4. Evaluation & Inspection

**Purpose.** Evaluate **`best.pt`** (not `last.pt`) and inspect beyond a single scalar.

`best.pt` is the epoch with best validation fitness; `last.pt` is only for resume. `val()` uses a low conf threshold so the full PR curve can be computed; pick a deployment conf later from the F1 curve.

**Edit:** rarely anything — mainly inspect.


In [ ]:
BEST = RUN_DIR / "weights" / "best.pt"
best_model = YOLO(BEST)

metrics = best_model.val(data=DATA, plots=True, save_json=True)
print(metrics.results_dict)

**How to read the plots:**

- **`results.png`** — train/val loss and val mAP. Falling train loss + flat/dropping val mAP → overfitting; both flat and mediocre → underfitting.
- **`confusion_matrix.png`** — off-diagonal clusters = similar classes or inconsistent labels; heavy background row = missed objects; heavy background column = false positives.
- **`PR_curve.png`** — area under curve is mAP; early collapse means few high-confidence correct predictions.
- **`F1_curve.png`** — peak marks a practical deployment conf threshold.
- **`labels.jpg`** — class balance (same as Section 2).


In [ ]:
for plot_name in (
    "labels.jpg",
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "F1_curve.png",
):
    plot_path = RUN_DIR / plot_name
    if plot_path.exists():
        display(Image(filename=str(plot_path)))

**Why look at raw predictions?** Aggregate mAP can hide systematic failures on a subset (lighting, orientation, rare class). Skimming predicted images is the cheapest way to find those modes.


In [ ]:
import cv2
from PIL import Image as PILImage

val_dir = dataset_info["val"] if dataset_info else DATA
val_dir = val_dir[0] if isinstance(val_dir, list) else val_dir

predictions = best_model.predict(source=val_dir, conf=0.25, verbose=False)
for r in predictions[:5]:
    display(PILImage.fromarray(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)))

## 5. Experiment Summary

**Purpose.** One readable record of config + metrics (assembled from `args.yaml` / `results.csv` under `RUN_DIR`). Read-only; works even if you re-run only this cell later. If you ran optional tuning, note the path to `best_hyperparameters.yaml` in the free-form notes of the report (Section 7).


In [ ]:
import pandas as pd

run_args = yaml.safe_load(open(RUN_DIR / "args.yaml"))
results_csv = pd.read_csv(RUN_DIR / "results.csv")
results_csv.columns = [c.strip() for c in results_csv.columns]

if "metrics" not in dir():
    best_model = YOLO(RUN_DIR / "weights" / "best.pt")
    metrics = best_model.val(data=DATA)

summary = {
    "experiment": EXPERIMENT_NAME,
    "task": TASK,
    "model": MODEL,
    "data": DATA,
    "epochs_ran": int(results_csv["epoch"].iloc[-1]) + 1,
    "overrides": overrides,
    "final_metrics": metrics.results_dict,
    "run_dir": str(RUN_DIR),
}

for key, value in summary.items():
    print(f"{key}: {value}")

## 6. Export

**Purpose.** Convert to a deployment format and confirm accuracy did not silently drop.

Choose format by target runtime (e.g. TensorRT/`engine`, CoreML, ONNX). `quantize` (16/8) trades size and speed for accuracy — benchmark on the real target.

**Edit:** `EXPORT_FORMAT` and any extra export args (see `yolo-export` skill).


In [ ]:
EXPORT_FORMAT = "onnx"  # torchscript | onnx | openvino | engine | coreml | ...

export_path = best_model.export(format=EXPORT_FORMAT)
print(f"Exported to: {export_path}")

**Why re-validate after export?** Conversion changes precision and operators; it is not lossless. Run the same `val()` on the exported artifact before production.


In [ ]:
exported_model = YOLO(export_path)
exported_metrics = exported_model.val(data=DATA).results_dict

print("Baseline (.pt):", summary["final_metrics"])
print("Exported:      ", exported_metrics)

## 7. Report

**Purpose.** Write a durable markdown report under `docs/{EXPERIMENT_NAME}_report.md` (config, summary, metrics, export, free-form notes). Survives kernel restarts and makes the experiment reproducible for others (or future you).


In [ ]:
report_path = Path("docs") / f"{EXPERIMENT_NAME}_report.md"
report_path.parent.mkdir(parents=True, exist_ok=True)

report = f"""# Experiment Report — {EXPERIMENT_NAME}

## Configuration
- Task: {TASK}
- Model: {MODEL}
- Data: {DATA}
- Overrides: {overrides}

## Training
- Run directory: `{RUN_DIR}`
- Epochs ran: {summary['epochs_ran']}

## Optional tuning
- Ran: {RUN_TUNE}
- (If True) best hyperparameters file: `runs/{TASK}/{EXPERIMENT_NAME}_tune/best_hyperparameters.yaml`
- Promote useful values into a new experiment YAML and retrain fully; do not ship the short-search run.

## Evaluation
- Final metrics: {summary['final_metrics']}

## Export
- Format: {EXPORT_FORMAT}
- Artifact: `{export_path}`
- Exported metrics: {exported_metrics}

## Notes
_Add qualitative observations, failure cases, and next steps here._
"""

report_path.write_text(report)
print(f"Report written to {report_path}")

---

## Adding more experiments to this project

A project is task + dataset; keep multiple experiments inside it.

1. Change `MODEL` and/or add `configs/experiments/<name>.yaml`.
2. Set a new self-describing `EXPERIMENT_NAME`.
3. Re-run training + evaluation (or the whole notebook).

Runs stay under the same `runs/<task>/` for easy comparison.

## Common mistakes

| Mistake | Fix |
|---|---|
| Wrong `TASK` vs. model suffix | Use the Section 1 sanity check |
| Relative `path:` + stale `datasets_dir` | Set `datasets_dir` once per environment |
| Skipping the smoke test | Run once per new/changed dataset |
| Evaluating `last.pt` instead of `best.pt` | Always load `weights/best.pt` |
| Dumping every arg into overrides YAML | Record only intentional changes |
| New project per model variant | Keep experiments in one task+dataset project |
| Enabling `RUN_TUNE` before a healthy baseline | Always inspect Section 4 first; promote hyps then retrain |
| Adding `class_aliases` without a prior unaliased baseline | Always train `baseline` first; compare per-class metrics before keeping an aliased run |

## Checklist before calling an experiment done

- [ ] Dataset YAML/folder checked
- [ ] Visual spot-check and smoke test passed
- [ ] `TASK` matches model suffix
- [ ] Self-describing experiment name
- [ ] Overrides only intentional
- [ ] If tuning was used: promoted best hyps into a new experiment and retrained fully
- [ ] If class aliasing was used: compared per-class metrics against an unaliased baseline
- [ ] Evaluated `best.pt`
- [ ] Report under `docs/`
- [ ] Notebook still understandable in a fresh session

---
Skills: `yolo-models`, `yolo-datasets`, `yolo-training`, `yolo-tuning`, `yolo-inference`, `yolo-export`.  
Docs: `docs/02_how_to_edit_the_template.md`, `docs/03_notebook_walkthrough.md`.
